In [1]:
!pip install langchain langchain-community langchain-openai
!pip install faiss-cpu tiktoken
!pip install streamlit pyngrok
!pip install kagglehub pandas
!pip install sentence-transformers
!pip install langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

In [2]:
import kagglehub

path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'wikipedia-movie-plots' dataset.
Path to dataset files: /kaggle/input/wikipedia-movie-plots


In [3]:
import pandas as pd
import os

# Find the CSV file
for f in os.listdir(path):
    print(f)

df = pd.read_csv(os.path.join(path, "wiki_movie_plots_deduped.csv"))
print(df.shape)
df.head(3)

wiki_movie_plots_deduped.csv
(34886, 8)


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."


In [4]:
# Drop nulls and select relevant columns
df = df[['Title', 'Director', 'Genre', 'Plot']].dropna()

# Sample 500 movies to keep vector store manageable (increase if you have GPU)
df_sample = df.sample(500, random_state=42).reset_index(drop=True)

# Create a combined text field for embedding
df_sample['combined'] = (
    "Title: " + df_sample['Title'] + "\n" +
    "Director: " + df_sample['Director'] + "\n" +
    "Genre: " + df_sample['Genre'] + "\n" +
    "Plot: " + df_sample['Plot']
)

print(f"Sample size: {len(df_sample)}")
print(df_sample['combined'][0][:300])

Sample size: 500
Title: The Day the Earth Stood Still
Director: Robert Wise
Genre: science fiction
Plot: When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a


In [6]:
!pip install -qU langchain langchain-core langchain-community langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.9 MB/s eta 0:00:00


In [7]:
# Fix for newer LangChain versions (schema moved)
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.docstore.document import Document

docs = [
    Document(
        page_content=row['combined'],
        metadata={
            "title": row['Title'],
            "director": row['Director'],
            "genre": row['Genre']
        }
    )
    for _, row in df_sample.iterrows()
]

print(f"Total documents: {len(docs)}")
print(docs[0].page_content[:200])

Total documents: 500
Title: The Day the Earth Stood Still
Director: Robert Wise
Genre: science fiction
Plot: When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerg


In [9]:
!pip install -qU langchain-text-splitters

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(docs)
print(f"Total chunks after splitting: {len(split_docs)}")

Total chunks after splitting: 3575


In [10]:
!pip install -qU faiss-cpu sentence-transformers langchain-huggingface

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_community.vectorstores import FAISS
except ImportError:
    from langchain.vectorstores import FAISS

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Building FAISS vector store (this may take 1-2 mins)...")
vectorstore = FAISS.from_documents(split_docs, embeddings)

# Save locally for reuse
vectorstore.save_local("movie_faiss_index")
print("Vector store saved!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 14.9 MB/s eta 0:00:00


/tmp/ipykernel_2447/88523670.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS vector store (this may take 1-2 mins)...
Vector store saved!


In [11]:
query = "Tell me about a sci-fi movie with time travel"
results = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Title: {r.metadata['title']}")
    print(f"Genre: {r.metadata['genre']}")
    print(r.page_content[:200])


--- Result 1 ---
Title: Indru Netru Naalai
Genre: science-fiction comedy
Title: Indru Netru Naalai
Director: Ravi Kumar R.
Genre: science-fiction comedy
Plot: In the year 2065, a scientist (Arya) invents a time machine. To prove its capability, he sends it back in time to 

--- Result 2 ---
Title: The Joy Luck Club
Genre: drama
the time the film is set. The mothers have high hopes for their daughters' success, but the daughters struggle through "anxieties, feelings of inadequacy, and failures." Throughout the film, the mothe

--- Result 3 ---
Title: Indru Netru Naalai
Genre: science-fiction comedy
Since the time machine was never seen, it returned to 2065, making it a success.


In [18]:
!pip install -qU langchain-groq groq

import os
os.environ["GROQ_API_KEY"] = "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"

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ✅ updated model name
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.5,
    max_tokens=512
)

# Quick test
response = llm.invoke("Name a good horror movie")
print(response.content)
print("✅ LLM loaded!")

One highly-regarded horror movie is "The Shining" (1980) directed by Stanley Kubrick. It's an adaptation of Stephen King's novel of the same name and stars Jack Nicholson as a writer who becomes unhinged while caring for a haunted hotel. The film's atmospheric tension, eerie setting, and unsettling performances make it a classic horror movie that continues to terrify audiences to this day.
✅ LLM loaded!


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Load vector store
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "movie_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# ── Improved Prompt with explicit memory instruction ───────────
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable movie expert chatbot with a great memory.

IMPORTANT RULES:
1. Always remember what movies you mentioned in previous messages
2. When asked follow-up questions like "which of those", "who directed it",
   "tell me more" — refer back to the movies you already mentioned
3. Use the retrieved context below to answer accurately
4. Always mention the director and genre when discussing a movie

Retrieved Movie Information:
{context}

Previous conversation is in the chat history below. Always refer to it for follow-up questions."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Format docs
def format_docs(docs):
    return "\n\n".join([
        f"Title: {d.metadata['title']}\n"
        f"Director: {d.metadata['director']}\n"
        f"Genre: {d.metadata['genre']}\n"
        f"Plot: {d.page_content}"
        for d in docs
    ])

# Chat history list
chat_history = []

# RAG chain
rag_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x["chat_history"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG chain ready!


In [22]:
def chat(question):
    global chat_history

    result = rag_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    source_docs = retriever.invoke(question)

    # Save to memory
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result))

    print(f"\n🎬 Question: {question}")
    print(f"\n🤖 Answer:\n{result}")
    print("\n📚 Sources:")
    for doc in source_docs[:2]:
        print(f"  - {doc.metadata['title']} "
              f"| Dir: {doc.metadata['director']} "
              f"| Genre: {doc.metadata['genre']}")
    print("-" * 60)

# ── Reset and test ─────────────────────────────────────────────
chat_history = []

chat("What are some good horror movies with scary plots?")
chat("Which of those has the scariest plot? Give details.")
chat("Who directed that movie and what year was it made?")


🎬 Question: What are some good horror movies with scary plots?

🤖 Answer:
I can suggest a few horror movies with scary plots from my knowledge. 

One movie that comes to mind is Night of the Living Dead 3D, a 3D horror film directed by Jeff Broadstreet. It's a remake of the 1968 classic and has a creepy atmosphere.

Another movie that's sure to send chills down your spine is Ghost Ship, a horror film directed by Steve Beck. It's a supernatural horror movie that's set on a haunted ship.

Lastly, I'd recommend The Final Destination, a horror film directed by David R. Ellis. It's a thrilling movie that's all about predicting and avoiding death.

All three movies have intense and scary plots that are sure to keep you on the edge of your seat.

Would you like to know more about any of these movies or is there something else I can help you with?

📚 Sources:
  - Night of the Living Dead 3D | Dir: Jeff Broadstreet | Genre: horror
  -  The Scarecrow | Dir: Richard Rich | Genre: animation, fant